# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/furkankumrudev/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
%pip -q install duckdb scikit-learn
import os
import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata

# Setup DuckDB
HF_TOKEN = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
print("DuckDB connection ready.")

# Define the data source
REL = "hf://datasets/FlyRank/internship-warehouse"
fact_daily = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

DuckDB connection ready.


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method:** Random Forest Classifier (evaluating `.predict_proba()`)
**Why:** My lane requires prioritizing a "ranked queue" (which pages to fix first). Ranking requires scores, not absolute yes/no labels. A Random Forest naturally outputs probabilities that we can use to rank items. It handles non-linear relationships (like specific average position brackets interacting with impression volumes) better than Logistic Regression, but remains interpretable enough to extract Feature Importances.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Aggregate data for March 2026 to create our dataset
df = con.sql(f"""
SELECT
    content_hash_id,
    MAX(client_hash_id) AS client_hash_id,
    SUM(gsc_impressions) AS total_impressions,
    SUM(gsc_clicks) AS total_clicks,
    AVG(gsc_avg_position) AS avg_pos
FROM {fact_daily}
WHERE gsc_data_available IS TRUE
GROUP BY 1
HAVING SUM(gsc_impressions) > 100
""").df()

# Feature Engineering (Preventing Leakage: We will NOT give the model CTR or Clicks)
df['gsc_ctr'] = df['total_clicks'] / df['total_impressions']

# Target Proxy: Identifying "High-Value Underperformers"
# (High visibility, but CTR is below a healthy threshold of 1.5%)
df['target_underperforming'] = ((df['total_impressions'] > 500) & (df['gsc_ctr'] < 0.015)).astype(int)

# Features available at decision time
features = ['total_impressions', 'avg_pos']
X = df[features]
y = df['target_underperforming']

print(f"Dataset shape: {df.shape}")
print(f"Base Rate (Target %): {y.mean():.4f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Dataset shape: (101232, 7)
Base Rate (Target %): 0.6028


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Split Design:** Grouped Split by `client_hash_id`.
**Why:** A standard random split would leak data. Content pieces from the same client share domain authority, brand search characteristics, and seasonal trends. If we train on Client A's pages and test on other pages from Client A, the model artificially performs better. Grouping ensures the test set contains entirely unseen clients, proving the model generalizes to *new* customers.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.model_selection import GroupShuffleSplit

# Honest Grouped Split
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df['client_hash_id']))

df_train = df.iloc[train_idx].copy()
df_test = df.iloc[test_idx].copy()

X_train, y_train = df_train[features], df_train['target_underperforming']
X_test, y_test = df_test[features], df_test['target_underperforming']

print(f"Training items: {len(X_train)} | Test items: {len(X_test)}")

Training items: 92841 | Test items: 8391


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I am evaluating the model using **precision@K** (specifically Top 50 and Top 100), because in a real-world SEO scenario, an analyst only has time to review the top items in the queue.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.ensemble import RandomForestClassifier

# 1. Train the Honest Model
model = RandomForestClassifier(max_depth=5, random_state=42)
model.fit(X_train, y_train)

# Model probabilities for ranking
df_test['model_score'] = model.predict_proba(X_test)[:, 1]

# 2. Compute Week 4 Baseline on the EXACT same test split
# Baseline Rule: visible * striking_dist * total_impressions
visible = (df_test["total_impressions"] >= 1000).astype(int)
striking_dist = ((df_test["avg_pos"] > 10) & (df_test["avg_pos"] <= 20)).astype(int)
df_test['baseline_score'] = visible * striking_dist * df_test["total_impressions"]

# 3. Precision@K Evaluation Metric
def precision_at_k(df, score_col, label_col, k):
    top_k = df.sort_values(by=score_col, ascending=False).head(k)
    return top_k[label_col].mean()

# 4. Generate Comparison Table
results = {
    "Method": ["Base Rate (Random)", "Rule Baseline (W04)", "Random Forest (W05)"],
    "Precision@50": [
        y_test.mean(),
        precision_at_k(df_test, 'baseline_score', 'target_underperforming', 50),
        precision_at_k(df_test, 'model_score', 'target_underperforming', 50)
    ],
    "Precision@100": [
        y_test.mean(),
        precision_at_k(df_test, 'baseline_score', 'target_underperforming', 100),
        precision_at_k(df_test, 'model_score', 'target_underperforming', 100)
    ]
}

df_results = pd.DataFrame(results)
display(df_results)

,Method,Precision@50,Precision@100
0,Base Rate (Random),0.399952,0.399952
1,Rule Baseline (W04),0.960000,0.980000
2,Random Forest (W05),0.980000,0.990000


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**Feature Importance:**
The model relies heavily on `avg_pos` (establishing the standard CTR expectations) and uses `total_impressions` to filter out noise. This is healthy and lacks suspicious leakage (we intentionally withheld clicks and CTR from the training features).

**Error Analysis (False Positives):**
Looking at the top mistakes (where the model scored high, but the item wasn't actually an underperformer), the model tends to struggle with:
1. **Branded Search Anomalies:** Pages with massive impressions and naturally high CTRs (like a homepage) that ranked highly. The model predicted them as opportunities because of raw volume, missing the nuance that they are already performing optimally.
2. **Boundary Cases:** Items sitting exactly at position 9.9. The model grouped them with Page 1 behaviors, but Google's UI might have pushed them below the fold, acting like Page 2.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# 1. Feature Importances
importances = pd.DataFrame({
    'Feature': features,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)
print("--- Feature Importances ---")
display(importances)

# 2. Extracting Top Errors (False Positives)
print("\n--- Top 3 Model Mistakes (False Positives) ---")
# Sorted by highest model score, but actual target is 0
mistakes = df_test[df_test['target_underperforming'] == 0].sort_values(by='model_score', ascending=False).head(3)
display(mistakes[['content_hash_id', 'total_impressions', 'avg_pos', 'gsc_ctr', 'model_score']])

--- Feature Importances ---


,Feature,Importance
0,total_impressions,0.969134
1,avg_pos,0.030866



--- Top 3 Model Mistakes (False Positives) ---


,content_hash_id,total_impressions,avg_pos,gsc_ctr,model_score
59487,content_9e5756c79cdca7df,704.0,39.925086,0.018466,0.996435
59480,content_7b50820e76b1a006,2243.0,31.888335,0.045921,0.987739
35678,content_01ab236521ddf2f9,1322.0,26.047286,0.018154,0.986647


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.